# Tabla maestra — todas las combinaciones y todos los indicadores

Los resultados del proyecto están repartidos entre los logs, las predicciones fuera de muestra y los notebooks de evaluación, pruebas estadísticas y costo. La **tabla maestra** los junta en un solo archivo: una fila por cada una de las **136 combinaciones** (108 de clasificación y 28 de regresión) con su desempeño, su calibración, las pruebas contra la mejor de su tarea, la pertenencia al conjunto de confianza de modelos, la regla de 1 error estándar, el costo y los hiperparámetros típicos.

Se construye con `src/tabla_maestra.py` a partir de `experiments/logs/` y `experiments/predicciones/`, y se guarda en `experiments/tablas/tabla_maestra.csv` junto con su diccionario de columnas.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

RUTA_MODELADO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(RUTA_MODELADO) not in sys.path:
    sys.path.insert(0, str(RUTA_MODELADO))

from src.entorno import RUTA_ENTORNO, guardar_entorno, versiones
from src.graficos import guardar_tabla
from src.tabla_maestra import DESCRIPCION, RUTA_TABLA_MAESTRA, construir

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 250)
display(versiones().set_index("paquete").T)

paquete,python,numpy,pandas,scipy,scikit-learn,imbalanced-learn,xgboost,optuna,deap,joblib,statsmodels,scikit-posthocs,shap,lime,matplotlib,seaborn
versión,3.12.14,2.4.4,3.0.2,1.17.1,1.8.0,0.14.2,3.4.1,5.0.0,1.4.4,1.6.0,0.15.0,0.16.1,0.52.0,0.2.0.1,3.10.8,0.13.2


In [2]:
maestra = construir(alfa_mcs=0.10, B=2000)
maestra.to_csv(RUTA_TABLA_MAESTRA, index=False, float_format="%.6g")
diccionario = pd.DataFrame(list(DESCRIPCION.items()), columns=["columna", "descripcion"])
guardar_tabla(diccionario, "tabla_maestra_diccionario")

print(f"{len(maestra)} filas × {maestra.shape[1]} columnas → {RUTA_TABLA_MAESTRA.relative_to(RUTA_MODELADO)}")
display(maestra.groupby("tarea").agg(combinaciones=("modelo", "size"), modelos=("modelo", "nunique"),
                                     en_mcs=("en_mcs", "sum"), dentro_1ee=("dentro_1ee", "sum"),
                                     recomendadas=("recomendada", "sum"), horas_cpu=("horas_cpu", "sum")).round(2))
display(diccionario)

136 filas × 36 columnas → experiments\tablas\tabla_maestra.csv


,combinaciones,modelos,en_mcs,dentro_1ee,recomendadas,horas_cpu
tarea,,,,,,
clasificacion,108,7,8,26,1,13.51
regresion,28,7,1,6,1,3.07


,columna,descripcion
0,tarea,clasificacion (quality ≥ 7) o regresion (quality)
1,modelo / balanceo / optimizador,la combinación evaluada
2,rango,puesto dentro de su tarea según la métrica principal (AUC o RMSE; 1 = mejor)
3,"auc, f1, recall, precision, accuracy",media de los 5 folds externos (umbral 0.5); *_std = desviación entre folds
4,"rmse, mae, r2",media de los 5 folds externos; *_std = desviación entre folds
5,score_interno,mejor score de la validación interna (AUC o RMSE) promediado entre folds
6,brecha_interno_externo,métrica externa − interna (AUC: negativo = optimista; RMSE: positivo = optimista)
7,"brier, ece, pr_auc","calculadas con las 5,320 predicciones fuera de muestra (vacías para SVM, que no da pro..."
8,dif_vs_mejor,AUC (o error cuadrático medio) de la combinación menos el de la mejor de su tarea
9,p_vs_mejor_holm,"p-valor de DeLong (clasificación) o Diebold-Mariano (regresión) contra la mejor, ajust..."


## 1. Clasificación: las 15 mejores combinaciones por AUC

Columnas principales (la tabla completa tiene 36). `p_vs_mejor_holm` es el p-valor de DeLong contra la combinación de mayor AUC, ajustado por Holm sobre las 107 comparaciones; `p_mcs` y `en_mcs` vienen del conjunto de confianza de modelos con pérdida de Brier (vacíos en SVM, que no entrega probabilidades).

In [3]:
COLUMNAS_CLF = ["rango", "modelo", "balanceo", "optimizador", "auc", "auc_std", "f1", "recall", "precision", "brier",
                "ece", "dif_vs_mejor", "p_vs_mejor_holm", "p_mcs", "en_mcs", "dentro_1ee", "recomendada", "horas_cpu"]
clasificacion = maestra[maestra["tarea"] == "clasificacion"]
display(clasificacion[COLUMNAS_CLF].head(15).round(4).reset_index(drop=True))

,rango,modelo,balanceo,optimizador,auc,auc_std,f1,recall,precision,brier,ece,dif_vs_mejor,p_vs_mejor_holm,p_mcs,en_mcs,dentro_1ee,recomendada,horas_cpu
0,1,Random Forest,class_weight,Grid Search,0.8605,0.0152,0.4494,0.3390,0.6700,0.1090,0.0137,0.0000,NaN,1.0000,True,True,False,0.1634
1,2,Random Forest,class_weight,Bayesiano,0.8602,0.0146,0.4799,0.3846,0.6534,0.1093,0.0124,-0.0015,0.2908,0.3635,True,True,False,0.3628
2,3,Random Forest,class_weight,Random Search,0.8591,0.0138,0.5193,0.4540,0.6130,0.1107,0.0325,-0.0028,0.1451,0.1495,True,True,False,0.3035
3,4,Random Forest,SMOTE,Grid Search,0.8581,0.0157,0.5660,0.6016,0.5353,0.1204,0.0877,-0.0026,0.2908,0.0000,False,True,False,0.3038
4,5,Random Forest,class_weight,Genético,0.8579,0.0128,0.5018,0.4532,0.6097,0.1135,0.0351,-0.0101,0.0000,0.0065,False,True,False,0.3256
5,6,XGBoost,SMOTE,Bayesiano,0.8578,0.0162,0.5548,0.5917,0.5241,0.1192,0.0666,-0.0037,0.2908,0.0000,False,True,False,0.5206
6,7,Random Forest,SMOTE,Bayesiano,0.8575,0.0156,0.5613,0.6115,0.5203,0.1220,0.0941,-0.0031,0.1837,0.0000,False,True,False,0.5768
7,8,XGBoost,class_weight,Bayesiano,0.8575,0.0128,0.5592,0.6135,0.5170,0.1229,0.0680,-0.0054,0.0587,0.0000,False,True,False,0.2656
8,9,Random Forest,ADASYN,Grid Search,0.8575,0.0177,0.5676,0.6294,0.5176,0.1238,0.0999,-0.0032,0.2908,0.0000,False,True,False,0.3079
9,10,Random Forest,SMOTE,Genético,0.8568,0.0154,0.5622,0.6185,0.5173,0.1232,0.0969,-0.0041,0.0587,0.0000,False,True,False,0.4791


## 2. Regresión: las 28 combinaciones

`p_vs_mejor_holm` es el p-valor de Diebold–Mariano (error cuadrático) contra la combinación de menor RMSE, ajustado por Holm; el MCS usa el error cuadrático de cada vino.

In [4]:
COLUMNAS_REG = ["rango", "modelo", "optimizador", "rmse", "rmse_std", "mae", "r2", "score_interno", "brecha_interno_externo",
                "dif_vs_mejor", "p_vs_mejor_holm", "p_mcs", "en_mcs", "dentro_1ee", "recomendada", "horas_cpu",
                "hiperparametros_tipicos"]
regresion = maestra[maestra["tarea"] == "regresion"]
display(regresion[COLUMNAS_REG].round(4).reset_index(drop=True))

,rango,modelo,optimizador,rmse,rmse_std,mae,r2,score_interno,brecha_interno_externo,dif_vs_mejor,p_vs_mejor_holm,p_mcs,en_mcs,dentro_1ee,recomendada,horas_cpu,hiperparametros_tipicos
0,1,Random Forest,Grid Search,0.6787,0.0148,0.5226,0.4033,0.6903,-0.0116,0.0000,NaN,1.000,True,True,True,0.1711,"max_depth=25, max_features=sqrt, min_samples_leaf=1, n_estimators=500"
1,2,Random Forest,Genético,0.6805,0.0136,0.5241,0.4001,0.6903,-0.0098,0.0024,0.0053,0.043,False,True,False,0.3682,"n_estimators=443, max_depth=22, min_samples_leaf=2, max_features=sqrt"
2,3,Random Forest,Bayesiano,0.6812,0.0133,0.5258,0.3989,0.6897,-0.0085,0.0033,0.0015,0.043,False,True,False,0.3297,"n_estimators=398, max_depth=15, min_samples_leaf=1, max_features=log2 (3/5)"
3,4,Random Forest,Random Search,0.6818,0.0145,0.5263,0.3979,0.6904,-0.0086,0.0042,0.0001,0.043,False,True,False,0.3377,"max_depth=17, max_features=log2 (3/5), min_samples_leaf=1, n_estimators=459"
4,5,XGBoost,Bayesiano,0.6842,0.0186,0.5291,0.3935,0.6901,-0.0058,0.0077,0.0015,0.043,False,True,False,0.4145,"n_estimators=479, max_depth=8, learning_rate=0.0115, subsample=0.57, colsample_bytree=..."
5,6,XGBoost,Random Search,0.6852,0.0125,0.5283,0.3918,0.6918,-0.0066,0.0088,0.0006,0.019,False,True,False,0.2921,"colsample_bytree=0.801, learning_rate=0.0158, max_depth=7, n_estimators=322, subsample..."
6,7,XGBoost,Genético,0.6868,0.0182,0.5305,0.3890,0.6929,-0.0061,0.0112,0.0002,0.015,False,False,False,0.2827,"n_estimators=179, max_depth=7, learning_rate=0.026, subsample=0.571, colsample_bytree=..."
7,8,SVR,Bayesiano,0.6960,0.0132,0.5317,0.3725,0.6997,-0.0037,0.0237,0.0000,0.000,False,False,False,0.1621,"C=2.21, gamma=0.0784, epsilon=0.0426"
8,9,SVR,Random Search,0.6976,0.0167,0.5394,0.3695,0.7000,-0.0024,0.0261,0.0000,0.000,False,False,False,0.1326,"C=6.59, epsilon=0.206, gamma=0.0294"
9,10,SVR,Genético,0.6980,0.0124,0.5351,0.3687,0.7001,-0.0021,0.0265,0.0000,0.000,False,False,False,0.1598,"C=1.93, gamma=0.072, epsilon=0.139"


## 3. Resumen por modelo

Para leer la tabla de un vistazo, una fila por modelo y tarea: la combinación con mejor score interno (la que se elegiría sin mirar la prueba externa), con sus indicadores principales. Esta es la versión que se muestra en el informe.

In [5]:
orden_interno = np.where(maestra["tarea"] == "clasificacion", -maestra["score_interno"], maestra["score_interno"])
resumen = (maestra.assign(orden_interno=orden_interno).sort_values("orden_interno")
           .drop_duplicates(["tarea", "modelo"]).sort_values(["tarea", "rango"]).reset_index(drop=True))
horas_modelo = maestra.groupby(["tarea", "modelo"])["horas_cpu"].sum().rename("horas_cpu_modelo").reset_index()
resumen = resumen.merge(horas_modelo, on=["tarea", "modelo"])
COLUMNAS_RESUMEN = ["tarea", "modelo", "balanceo", "optimizador", "rango", "auc", "f1", "brier", "ece", "rmse", "mae", "r2",
                    "dif_vs_mejor", "p_vs_mejor_holm", "en_mcs", "dentro_1ee", "horas_cpu_modelo"]
tabla_resumen = resumen[COLUMNAS_RESUMEN]
guardar_tabla(tabla_resumen.round(5), "tabla_maestra_resumen_por_modelo")
display(tabla_resumen.round(3))

,tarea,modelo,balanceo,optimizador,rango,auc,f1,brier,ece,rmse,mae,r2,dif_vs_mejor,p_vs_mejor_holm,en_mcs,dentro_1ee,horas_cpu_modelo
0,clasificacion,Random Forest,class_weight,Bayesiano,2,0.860,0.480,0.109,0.012,NaN,NaN,NaN,-0.002,0.291,True,True,6.247
1,clasificacion,XGBoost,class_weight,Bayesiano,8,0.857,0.559,0.123,0.068,NaN,NaN,NaN,-0.005,0.059,False,True,4.119
2,clasificacion,SVM,class_weight,Bayesiano,34,0.847,0.548,NaN,NaN,NaN,NaN,NaN,-0.014,0.002,NaN,False,2.314
3,clasificacion,KNN,sin_balanceo,Bayesiano,36,0.845,0.436,0.114,0.011,NaN,NaN,NaN,-0.016,0.000,False,False,0.373
4,clasificacion,Regresión Logística (L1/L2),sin_balanceo,Bayesiano,57,0.828,0.384,0.119,0.025,NaN,NaN,NaN,-0.032,0.000,False,False,0.194
5,clasificacion,Decision Tree,class_weight,Bayesiano,83,0.801,0.505,0.183,0.192,NaN,NaN,NaN,-0.060,0.000,False,False,0.158
6,clasificacion,Naive Bayes,class_weight,Grid Search,93,0.765,0.444,0.285,0.305,NaN,NaN,NaN,-0.096,0.000,False,False,0.105
7,regresion,Random Forest,no_aplica,Bayesiano,3,NaN,NaN,NaN,NaN,0.681,0.526,0.399,0.003,0.002,False,True,1.207
8,regresion,XGBoost,no_aplica,Bayesiano,5,NaN,NaN,NaN,NaN,0.684,0.529,0.393,0.008,0.002,False,True,1.189
9,regresion,SVR,no_aplica,Bayesiano,8,NaN,NaN,NaN,NaN,0.696,0.532,0.372,0.024,0.000,False,False,0.548


## 4. Comprobaciones de consistencia

La tabla maestra debe coincidir con lo reportado en los otros notebooks: mismas combinaciones recomendadas, mismo número de configuraciones en el MCS y mismo tiempo total.

In [6]:
recomendadas = maestra[maestra["recomendada"]][["tarea", "modelo", "balanceo", "optimizador", "auc", "f1", "rmse", "r2"]]
display(recomendadas.round(4))
comprobaciones = pd.Series({
    "136 combinaciones (108 + 28)": len(maestra) == 136 and (maestra["tarea"] == "clasificacion").sum() == 108,
    "una recomendada por tarea": maestra.groupby("tarea")["recomendada"].sum().eq(1).all(),
    "la recomendada está dentro de 1 e.e.": maestra.loc[maestra["recomendada"], "dentro_1ee"].all(),
    "rango 1 tiene p_mcs = 1 en regresión": bool(regresion.loc[regresion["rango"] == 1, "p_mcs"].iloc[0] == 1),
    "SVM sin métricas de probabilidad": maestra.loc[maestra["modelo"] == "SVM", "brier"].isna().all(),
    "horas de CPU = 16.6": round(maestra["horas_cpu"].sum(), 1) == 16.6,
})
display(comprobaciones.to_frame("cumple"))

,tarea,modelo,balanceo,optimizador,auc,f1,rmse,r2
21,clasificacion,Random Forest,ADASYN,Random Search,0.8548,0.5751,NaN,NaN
108,regresion,Random Forest,no_aplica,Grid Search,NaN,NaN,0.6787,0.4033


,cumple
136 combinaciones (108 + 28),True
una recomendada por tarea,True
la recomendada está dentro de 1 e.e.,True
rango 1 tiene p_mcs = 1 en regresión,True
SVM sin métricas de probabilidad,True
horas de CPU = 16.6,True


## 5. Entorno

La lista completa de paquetes con sus versiones exactas se guarda en `experiments/entorno.txt`, para poder reconstruir el entorno con el que se generaron todas las tablas y figuras (además de `requirements.txt` y `environment.yml`).

In [7]:
guardar_entorno()
lineas = RUTA_ENTORNO.read_text(encoding="utf-8").splitlines()
print(f"{RUTA_ENTORNO.relative_to(RUTA_MODELADO)}: {sum(not l.startswith('#') for l in lineas)} paquetes")
print("\n".join(lineas[:8]))

experiments\entorno.txt: 80 paquetes
# Python 3.12.14
# sistema: Windows-11-10.0.26200-SP0
# procesador: AMD Ryzen 5 5600GT with Radeon Graphics
# núcleos lógicos: 12
# núcleos físicos: 6
# memoria RAM (GB): 15.8
# procesos en paralelo (N_WORKERS): 11
alembic==1.20.0


> **Lectura de la tabla maestra:** en clasificación, las primeras posiciones por AUC son casi todas de Random Forest y XGBoost, y las diferencias con la mejor son de milésimas (el p-valor de DeLong ajustado por Holm no rechaza en la mayoría de ellas). Las columnas de calibración separan dos grupos que el AUC no distingue: las mejores combinaciones de Random Forest con `class_weight` o sin balanceo tienen Brier ≈ 0.109–0.111 y entran al MCS, mientras que las de SMOTE y ADASYN (y XGBoost con `scale_pos_weight`) tienen Brier de 0.12 a 0.13 y quedan fuera. La configuración recomendada por la regla de 1 error estándar (Random Forest + ADASYN + Random Search) está dentro del error estándar en AUC y tiene el mejor F1, pero no está en el MCS porque sus probabilidades están infladas; si se necesitan probabilidades, la tabla apunta a Random Forest con `class_weight`. En regresión, Random Forest con Grid Search es a la vez la de menor RMSE, la recomendada y la única en el MCS.